# HYPER-POLY OPENWORLD FILM - Kaggle Render Pipeline
**30-minute procedural film rendered in chunks across Kaggle GPU sessions**

| Setting | Value |
|---------|-------|
| Frames | 21,600 @ 12fps |
| Resolution | 1280x720 |
| Engine | Cycles GPU (CUDA/OptiX) |
| Samples | 64 (adaptive) |
| GPU | P100 (3,584 CUDA / 732 GB/s / 9.3 TFLOPS) |
| Est. per frame | ~5-9s on P100 (~2x T4) |
| Est. total | ~30-55 GPU hours |
| Sessions needed | ~4-7 sessions @ 9h each |

### Kaggle Kernel Push - GPU Selection:
Include at top level of push payload: `"acceleratorTypeGPU": "teslaP100"` (also accepts `"acceleratorTypeGpu"`)
Without this field, Kaggle defaults to T4. 2-session GPU limit applies.

In [ ]:
import os, sys, subprocess, json, time, glob
from pathlib import Path
import shutil

WORK_DIR = Path('/kaggle/working')
RENDER_DIR = WORK_DIR / 'render_output'
RENDER_DIR.mkdir(exist_ok=True)
BLEND_FILE = WORK_DIR / 'film.blend'
CONFIG_FILE = WORK_DIR / 'film_config.json'
PROGRESS_FILE = WORK_DIR / 'render_progress.json'

# GPU check
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                         capture_output=True, text=True)
print(f"GPU: {gpu_info.stdout.strip()}")
print(f"CUDA available: {os.path.exists('/usr/local/cuda')}")

In [ ]:
# INSTALL BLENDER
BLENDER_VERSION = "4.1.1"
BLENDER_URL = f"https://download.blender.org/release/Blender4.1/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_DIR = Path('/kaggle/blender')
BLENDER_BIN = BLENDER_DIR / f'blender-{BLENDER_VERSION}-linux-x64' / 'blender'

if not BLENDER_BIN.exists():
    print(f"Downloading Blender {BLENDER_VERSION}...")
    !wget -q --show-progress {BLENDER_URL} -O /tmp/blender.tar.xz
    BLENDER_DIR.mkdir(exist_ok=True)
    !tar -xf /tmp/blender.tar.xz -C {BLENDER_DIR}
    print("Blender installed.")
else:
    print(f"Blender already at {BLENDER_BIN}")

result = subprocess.run([str(BLENDER_BIN), '--version'], capture_output=True, text=True)
print(result.stdout.split('\n')[0])

In [ ]:
# LOAD OR INITIALIZE PROGRESS (historical perf tracking)
SESSION_HOURS = 8.5
SESSION_SECONDS = SESSION_HOURS * 3600
SAFETY_MARGIN = 300  # 5 min buffer for checkpoint + ffmpeg
CALIBRATION_BATCH = 50
# P100 estimate (vs 12s for T4)
DEFAULT_SPF = 7.0

if PROGRESS_FILE.exists():
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)
    history = progress.get('completed_chunks', [])
    if history:
        total_sec = sum(c['elapsed_seconds'] for c in history)
        total_frames = sum(c['frames_rendered'] for c in history)
        hist_avg = total_sec / total_frames
        print(f"Historical avg: {hist_avg:.1f}s/frame ({total_frames} frames across {len(history)} chunks)")
    else:
        hist_avg = None
    print(f"Resuming from frame {progress['last_rendered']} / {progress['total_frames']}")
else:
    FPS = 12
    DURATION_MINUTES = 30
    TOTAL_FRAMES = FPS * 60 * DURATION_MINUTES
    progress = {
        'total_frames': TOTAL_FRAMES,
        'last_rendered': -1,
        'completed_chunks': [],
        'fps': FPS,
        'session_start': time.time(),
        'calibrated_sec_per_frame': None,
        'gpu_type': None,
    }
    hist_avg = None
    print(f"New render: {TOTAL_FRAMES} frames @ {FPS}fps = {DURATION_MINUTES}min")

# Detect GPU type for logging
if not progress.get('gpu_type'):
    gpu_name = gpu_info.stdout.strip()
    progress['gpu_type'] = gpu_name
    print(f"GPU: {gpu_name}")

EST_SECONDS_PER_FRAME = hist_avg if hist_avg else DEFAULT_SPF
MAX_FRAMES_THIS_SESSION = int((SESSION_SECONDS - SAFETY_MARGIN) / EST_SECONDS_PER_FRAME)
print(f"Initial estimate: {EST_SECONDS_PER_FRAME:.1f}s/fr -> plan up to {MAX_FRAMES_THIS_SESSION} frames")

In [ ]:
# BUILD BLENDER SCENE
if not BLEND_FILE.exists():
    print("Building film scene in Blender...")
    FILM_SCRIPT = Path('/kaggle/input/fractype-film/film.py')
    if not FILM_SCRIPT.exists():
        print("ERROR: film.py not found. Upload to /kaggle/input/fractype-film/film.py")
        sys.exit(1)
    shutil.copy(FILM_SCRIPT, WORK_DIR / 'film.py')
    cmd = [str(BLENDER_BIN), '--background', '--python', str(WORK_DIR / 'film.py'), '--python-exit-code', '1']
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(WORK_DIR))
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print(f"STDERR: {result.stderr[-2000:]}")
        raise RuntimeError("Scene build failed")
    save_cmd = [str(BLENDER_BIN), '--background', '--python-expr',
        f'import bpy; bpy.ops.wm.save_as_mainfile(filepath="{BLEND_FILE}"); print("SAVED")']
    subprocess.run(save_cmd, cwd=str(WORK_DIR))
    print(f"Scene saved to {BLEND_FILE}")
else:
    print(f"Using existing {BLEND_FILE}")

In [ ]:
# CALIBRATION BATCH - measure real render speed
def render_frames(start, end, tag=""):
    cmd = [
        str(BLENDER_BIN), '--background', str(BLEND_FILE),
        '--render-output', str(RENDER_DIR / 'frame_'),
        '--frame-start', str(start), '--frame-end', str(end),
        '--render-format', 'PNG', '--engine', 'CYCLES',
        '--render-anim', '--cycles-device', 'CUDA',
    ]
    print(f"  [{tag}] frames {start} -> {end} ({end - start + 1} frames)...")
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=False, cwd=str(WORK_DIR))
    elapsed = time.time() - t0
    return elapsed, result.returncode == 0

chunk_start = progress['last_rendered'] + 1

if progress.get('calibrated_sec_per_frame') or (progress.get('completed_chunks') and len(progress['completed_chunks']) >= 2):
    if progress.get('calibrated_sec_per_frame'):
        calibrated_spf = progress['calibrated_sec_per_frame']
        print(f"Using stored calibration: {calibrated_spf:.1f}s/frame")
    else:
        history = progress['completed_chunks']
        calibrated_spf = sum(c['elapsed_seconds'] for c in history) / sum(c['frames_rendered'] for c in history)
        print(f"Recalculated from history: {calibrated_spf:.1f}s/frame")
else:
    calib_frames = min(CALIBRATION_BATCH, progress['total_frames'] - chunk_start)
    if calib_frames < 5:
        calibrated_spf = EST_SECONDS_PER_FRAME
        print(f"Too few frames for calibration, using estimate: {calibrated_spf:.1f}s/frame")
    else:
        print(f"Running calibration batch ({calib_frames} frames)...")
        calib_end = chunk_start + calib_frames - 1
        calib_elapsed, ok = render_frames(chunk_start, calib_end, tag="CALIB")
        if not ok:
            raise RuntimeError("Calibration render failed")
        calibrated_spf = calib_elapsed / calib_frames
        print(f"Calibration: {calibrated_spf:.1f}s/frame (vs estimate {EST_SECONDS_PER_FRAME:.1f}s)")
        progress['calibrated_sec_per_frame'] = calibrated_spf
        progress['completed_chunks'].append({
            'start': chunk_start, 'end': calib_end,
            'frames_rendered': calib_frames,
            'elapsed_seconds': calib_elapsed, 'avg_spf': calibrated_spf,
            'type': 'calibration'
        })
        progress['last_rendered'] = calib_end
        chunk_start = calib_end + 1
        session_elapsed = calib_elapsed + (time.time() - progress.get('session_start', time.time()))
        remaining_sec = SESSION_SECONDS - session_elapsed - SAFETY_MARGIN
        MAX_FRAMES_THIS_SESSION = max(50, int(remaining_sec / calibrated_spf))
        print(f"ADAPTIVE RESCALE: {MAX_FRAMES_THIS_SESSION} frames remaining this session")
        with open(PROGRESS_FILE, 'w') as f:
            json.dump(progress, f, indent=2)

budgeted_end = min(chunk_start + MAX_FRAMES_THIS_SESSION - 1, progress['total_frames'] - 1)
if budgeted_end < chunk_start:
    print("No frames left in budget. Exiting cleanly.")
    sys.exit(0)
print(f"Render plan: {chunk_start} -> {budgeted_end} ({budgeted_end - chunk_start + 1} frames @ ~{calibrated_spf:.1f}s)")
print(f"Est. duration: {(budgeted_end - chunk_start + 1) * calibrated_spf / 60:.0f} min")

In [ ]:
# MAIN RENDER CHUNK
if budgeted_end >= chunk_start:
    print(f"RENDERING MAIN CHUNK: frames {chunk_start} -> {budgeted_end}")
    t0 = time.time()
    main_elapsed, ok = render_frames(chunk_start, budgeted_end, tag="MAIN")
    if not ok:
        raise RuntimeError("Main render failed. Check Blender output above.")
    frames_rendered = budgeted_end - chunk_start + 1
    actual_spf = main_elapsed / frames_rendered
    print(f"MAIN COMPLETE: {frames_rendered} frames in {main_elapsed:.0f}s ({actual_spf:.1f}s/frame)")
    progress['completed_chunks'].append({
        'start': chunk_start, 'end': budgeted_end,
        'frames_rendered': frames_rendered,
        'elapsed_seconds': main_elapsed, 'avg_spf': actual_spf,
        'type': 'main'
    })
    progress['last_rendered'] = budgeted_end
    if len(progress['completed_chunks']) >= 2:
        all_chunks = progress['completed_chunks']
        total_f = sum(c['frames_rendered'] for c in all_chunks)
        total_s = sum(c['elapsed_seconds'] for c in all_chunks)
        progress['calibrated_sec_per_frame'] = total_s / total_f
        print(f"Updated calibration: {progress['calibrated_sec_per_frame']:.1f}s/frame ({total_f} total frames)")
else:
    print("Skipping main render - budget exhausted.")

In [ ]:
# MID-SESSION DIVERGENCE CHECK
if progress.get('completed_chunks'):
    chunks = progress['completed_chunks']
    last_chunk = chunks[-1]
    last_spf = last_chunk['avg_spf']
    calib_spf = progress.get('calibrated_sec_per_frame', last_spf)
    divergence = abs(last_spf - calib_spf) / calib_spf * 100
    if divergence > 25:
        print(f"WARNING DIVERGENCE: last chunk {last_spf:.1f}s/fr vs calibrated {calib_spf:.1f}s/fr ({divergence:.0f}% off)")
        print("  Likely act transition - trees/fog/sun angle shifted render cost.")
        print("  Next session will recalibrate from this data.")
    else:
        print(f"Stable: last chunk within {divergence:.0f}% of calibration")

In [ ]:
# SAVE PROGRESS + CHECKPOINT
with open(PROGRESS_FILE, 'w') as f:
    json.dump(progress, f, indent=2)
rendered_files = sorted(glob.glob(str(RENDER_DIR / 'frame_*.png')))
print(f"Total frames on disk: {len(rendered_files)}")
print(f"Progress: {progress['last_rendered'] + 1} / {progress['total_frames']} ({100*(progress['last_rendered']+1)/progress['total_frames']:.1f}%)")
!mkdir -p /kaggle/working/film_checkpoint
!cp {PROGRESS_FILE} /kaggle/working/film_checkpoint/
!cp {BLEND_FILE} /kaggle/working/film_checkpoint/
print(f"Checkpoint saved to /kaggle/working/film_checkpoint/")

In [ ]:
# GENERATE PREVIEW VIDEO
def make_video(frames_dir, output_path, fps=12):
    cmd = ['ffmpeg', '-y', '-framerate', str(fps),
        '-pattern_type', 'glob', '-i', f'{frames_dir}/frame_*.png',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
        '-preset', 'medium', '-crf', '23', str(output_path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        size_mb = os.path.getsize(output_path) / (1024*1024)
        print(f"Video: {output_path} ({size_mb:.1f} MB)")
    else:
        print(f"ffmpeg error: {result.stderr[-500:]}")

rendered = sorted(glob.glob(str(RENDER_DIR / 'frame_*.png')))
if len(rendered) > 100:
    make_video(RENDER_DIR, WORK_DIR / 'film_preview.mp4', fps=progress['fps'])
else:
    print(f"Only {len(rendered)} frames - skipping video (need >100)")

## RESUME STRATEGY

### P100 Sessions (4-7 sessions total):
With P100 (~2x T4 throughput), expect 4,000-5,500 frames per 9h session.

### Adaptive chunk loop:
1. **Cell 3** - loads `render_progress.json`, seeds from historical avg sec/frame + GPU type
2. **Cell 5** - if no calibration yet, renders 50-frame batch, measures real P100 speed, rescales budget
3. **Cell 6** - renders remaining frames within recalibrated budget
4. **Cell 7** - divergence check: warns if act transition shifted cost >25%
5. **Cell 8** - saves checkpoint (blend + progress)

### Kaggle Kernel Push (GPU selection):
```json
{
  "acceleratorTypeGPU": "teslaP100",
  ...other push fields...
}
```
Both `acceleratorTypeGPU` and `acceleratorTypeGpu` accepted. Value: `"teslaP100"`.

### Dataset pipeline:
```
Session N output -> Create Dataset "film-checkpoint-v{N}"
Session N+1 input <- Attach Dataset "film-checkpoint-v{N}"
```

### Estimated sessions for full 30min on P100:
| Session | Frames | Film Time | Cumulative |
|---------|--------|-----------|------------|
| 1 | 0-5000 | 0:00-6:56 | 7 min |
| 2 | 5000-10000 | 6:56-13:52 | 14 min |
| 3 | 10000-15000 | 13:52-20:48 | 21 min |
| 4-5 | 15000-21600 | 20:48-30:00 | 30 min |

Half the sessions of T4. Divergence checks handle act transitions (forest scenes render slower than valley).